# Bài 1: Chơi Game với AI — Nhập môn Game AI & Reinforcement Learning

**Dựa theo:** Kaggle Learn — *Intro to Game AI and Reinforcement Learning*, bài "Play the Game"
(gốc: https://www.kaggle.com/code/alexisbcook/play-the-game)

**Người tổng hợp:** Notebook tiếng Việt, bao gồm lý thuyết + thực hành, có mở rộng liên hệ tới bài toán
điều khiển tín hiệu giao thông bằng RL (PPO/MAPPO).

---

## Mục tiêu bài học

Sau bài này, bạn sẽ:

1. Hiểu khái niệm **Game AI** và mối liên hệ với **Reinforcement Learning (RL)**.
2. Biết cách dùng thư viện `kaggle_environments` để tạo một **môi trường (environment)** game — cụ thể là **ConnectX** (game Connect 4 tổng quát hoá).
3. Biết cách viết một **agent** (hàm quyết định hành động) đơn giản.
4. Biết cách cho 2 agent đấu với nhau, xem lại ván đấu (replay), và tự chơi thử.
5. Hiểu các khái niệm nền tảng: `observation`, `action`, `config`, `reward` — đây chính là những khái niệm
   cốt lõi bạn sẽ dùng lại khi huấn luyện PPO/MAPPO cho đồ án điều khiển đèn giao thông.

> 💡 **Vì sao bài học này quan trọng cho đồ án của bạn?**
> ConnectX là một "sân chơi" (playground) rất tốt để làm quen với vòng lặp
> `environment → observation → agent → action → reward → environment mới...`
> Đây chính xác là vòng lặp bạn sẽ dùng khi PPO/MAPPO agent quan sát trạng thái giao lộ (mật độ xe, hàng đợi...)
> và ra quyết định pha đèn tín hiệu.

## Phần 1 — Lý thuyết

### 1.1. Game AI là gì?

**Game AI** là lĩnh vực nghiên cứu cách xây dựng các tác nhân (agent) có khả năng chơi game — từ các game
cờ bàn cổ điển (Cờ vua, Cờ vây, Connect 4...) đến các game điện tử phức tạp (StarCraft, Dota 2...).

Có nhiều cách tiếp cận để xây agent chơi game:

| Cách tiếp cận | Mô tả | Ví dụ |
|---|---|---|
| **Luật cố định (rule-based)** | Agent quyết định dựa trên các luật do con người viết sẵn (if-else) | Agent luôn đánh vào cột giữa |
| **Tìm kiếm (search-based)** | Agent "nhìn trước" vài nước đi rồi chọn nước tốt nhất (Minimax, Alpha-Beta, MCTS) | Agent one-step lookahead |
| **Học tăng cường (RL)** | Agent tự học chính sách (policy) tối ưu qua việc thử-sai và nhận thưởng | AlphaZero, PPO agent |

Bài học này bắt đầu từ mức đơn giản nhất: **agent rule-based**, để bạn làm quen với "bộ khung" (framework)
trước khi tiến tới các agent phức tạp hơn (tìm kiếm, rồi RL) ở các bài sau.

### 1.2. Reinforcement Learning — nhắc lại nhanh

RL mô hình hoá bài toán ra quyết định tuần tự bằng vòng lặp sau:

```
        action a_t
   ┌───────────────────►┌─────────────┐
   │                     │ Environment │
Agent                    │  (Môi trường)│
   │◄───────────────────┌│             │
        obs, reward      └─────────────┘
```

- **Environment (môi trường):** nơi "luật chơi" diễn ra — ở đây là bàn cờ ConnectX.
- **Agent (tác nhân):** thực thể ra quyết định — ở đây là hàm Python bạn viết.
- **Observation / State (quan sát/trạng thái):** thông tin agent nhận được về môi trường tại một thời điểm
  — ở đây là trạng thái bàn cờ.
- **Action (hành động):** lựa chọn agent đưa ra — ở đây là chọn cột để thả quân.
- **Reward (phần thưởng):** tín hiệu phản hồi môi trường trả về (thắng/thua/hoà) để đánh giá hành động.

> Trong ConnectX, ta *chưa* huấn luyện agent bằng RL ngay — ta chỉ **viết agent thủ công** để hiểu rõ
> "giao diện" (interface) mà agent phải tuân theo: nhận `obs` và `config`, trả về một `action` hợp lệ.
> Đây chính là interface mà sau này một PPO agent (mạng neural network) cũng phải tuân theo — chỉ khác là
> hành động được chọn bởi một policy network đã học, thay vì luật if-else.

### 1.3. Giới thiệu game ConnectX

**ConnectX** là bản tổng quát hoá của **Connect Four**:

- Bàn cờ có `rows` hàng × `columns` cột (mặc định 6×7 giống Connect Four).
- Hai người chơi lần lượt thả quân vào một cột; quân sẽ rơi xuống ô trống thấp nhất của cột đó.
- Người chơi nào xếp được `inarow` quân liên tiếp (ngang / dọc / chéo) trước thì thắng.
- Mặc định `inarow = 4` (giống Connect Four); nhưng ConnectX cho phép tuỳ chỉnh (ví dụ inarow=5 trên bàn to hơn).

### 1.4. Thư viện `kaggle_environments`

Đây là thư viện do Kaggle xây dựng để chuẩn hoá các môi trường thi đấu multi-agent (ConnectX, Rock-Paper-Scissors,
Hungry Geese...). Nó cung cấp:

- `make(tên_môi_trường)` — tạo môi trường.
- `env.reset()` — reset môi trường về trạng thái ban đầu.
- `env.render()` — hiển thị trạng thái hiện tại (dạng text hoặc HTML).
- `env.run([agent1, agent2])` — cho 2 agent tự động đấu 1 ván trọn vẹn.
- `env.play([...])` — cho phép người chơi thật tương tác qua giao diện.

## Phần 2 — Thực hành

### 2.1. Cài đặt thư viện

Chạy cell bên dưới để cài `kaggle_environments` (nếu chưa có). Nếu bạn chạy trên Kaggle Notebook thì thư viện
này đã có sẵn, có thể bỏ qua bước cài đặt.

In [ ]:
# Cài đặt thư viện (chỉ cần chạy 1 lần)
# Nếu chạy trên Kaggle Notebook, thư viện đã có sẵn -> có thể bỏ qua dòng dưới
!pip install kaggle_environments -q


In [ ]:
from kaggle_environments import make, evaluate
import numpy as np
import random

print("Import thành công!")


### 2.2. Tạo môi trường ConnectX

Ta dùng hàm `make("connectx")` để tạo môi trường. Tham số `debug=True` giúp in ra lỗi chi tiết nếu agent
của bạn bị crash (rất hữu ích khi debug).

In [ ]:
# Tạo môi trường ConnectX
env = make("connectx", debug=True)

# Xem cấu hình mặc định của môi trường (số hàng, số cột, số quân liên tiếp cần để thắng)
print("Cấu hình môi trường:", env.configuration)


**Giải thích `configuration`:**

- `columns`: số cột của bàn cờ (mặc định 7)
- `rows`: số hàng của bàn cờ (mặc định 6)
- `inarow`: số quân liên tiếp cần để thắng (mặc định 4)
- `timeout`: thời gian tối đa (giây) agent được phép "suy nghĩ" mỗi lượt
- `steps`: số lượt tối đa của cả ván

> 🔑 Đây chính là `config` mà hàm agent của bạn sẽ nhận được ở mỗi lượt đi.

### 2.3. Viết agent đầu tiên

Một **agent** trong `kaggle_environments` đơn giản là **một hàm Python** nhận vào 2 tham số:

```python
def my_agent(obs, config):
    ...
    return action  # số nguyên = chỉ số cột muốn thả quân (0-indexed)
```

Trong đó:

- **`obs`** (observation) có 2 thuộc tính quan trọng:
  - `obs.board`: một **list phẳng (flatten list)** độ dài `rows × columns`, biểu diễn bàn cờ.
    - `0` = ô trống
    - `1` = quân của người chơi 1
    - `2` = quân của người chơi 2
  - `obs.mark`: quân của **agent hiện tại** (giá trị `1` hoặc `2`)

- **`config`**: chính là `env.configuration` đã xem ở trên (`columns`, `rows`, `inarow`...)

Agent phải trả về một **số nguyên từ 0 đến `config.columns - 1`**, là chỉ số cột muốn thả quân vào.

Dưới đây là 3 agent đơn giản để bạn làm quen:

In [ ]:
# Agent 1: chọn cột hoàn toàn ngẫu nhiên trong số các cột còn trống
def agent_random(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return random.choice(valid_moves)


# Agent 2: luôn ưu tiên đánh vào cột chính giữa nếu còn trống, nếu không thì random
def agent_middle(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    mid = config.columns // 2
    if mid in valid_moves:
        return mid
    return random.choice(valid_moves)


# Agent 3: luôn đánh vào cột trống đầu tiên tính từ trái sang
def agent_leftmost(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return valid_moves[0]

print("Đã định nghĩa 3 agent: agent_random, agent_middle, agent_leftmost")


> 📝 **Lưu ý:** `obs.board[col] == 0` chỉ kiểm tra ô **trên cùng** của cột `col` (vì bàn cờ được lưu dạng
> flatten list, `rows` hàng đầu tiên trong list tương ứng với hàng trên cùng). Nếu ô trên cùng của cột còn
> trống (`0`), nghĩa là cột đó vẫn còn chỗ để thả quân xuống.

### 2.4. Cho hai agent đấu với nhau

Dùng `env.run([agent_1, agent_2])` để chạy trọn 1 ván. Agent đầu tiên trong list sẽ đi quân `1` (đi trước),
agent thứ hai đi quân `2`.

In [ ]:
# Chạy 1 ván giữa agent_middle (đi trước) và agent_random (đi sau)
env.run([agent_middle, agent_random])

# Hiển thị lại toàn bộ ván đấu dạng HTML (chạy tốt trên Jupyter/Kaggle Notebook)
env.render(mode="ipython", width=500, height=450)


Nếu `mode="ipython"` không hiển thị được (một số môi trường Jupyter không hỗ trợ), bạn có thể dùng
chế độ text:

In [ ]:
# Chế độ hiển thị dạng text (luôn hoạt động, kể cả trong terminal)
print(env.render(mode="ansi"))


### 2.5. Đánh giá agent qua nhiều ván đấu

Một ván đấu không đủ để kết luận agent nào "giỏi" hơn — vì có yếu tố ngẫu nhiên và lợi thế đi trước.
Ta nên cho 2 agent đấu **nhiều ván** (đổi cả thứ tự đi trước/sau) rồi tính **tỷ lệ thắng**.

Hàm `evaluate()` của `kaggle_environments` giúp làm việc này tự động.

In [ ]:
def get_win_percentages(agent1, agent2, n_rounds=100):
    """
    Cho agent1 và agent2 đấu n_rounds ván (chia đều đi trước/đi sau),
    rồi in ra tỷ lệ thắng/thua/hoà của từng agent.
    """
    config = {'rows': 6, 'columns': 7, 'inarow': 4}

    # Một nửa số ván: agent1 đi trước
    outcomes = evaluate("connectx", [agent1, agent2], config, [], n_rounds // 2)
    # Một nửa số ván: agent2 đi trước (đảo thứ tự kết quả để so sánh công bằng)
    outcomes += [[b, a] for [a, b] in evaluate("connectx", [agent2, agent1], config, [], n_rounds - n_rounds // 2)]

    win_1 = np.round(outcomes.count([1, -1]) / len(outcomes) * 100, 1)
    win_2 = np.round(outcomes.count([-1, 1]) / len(outcomes) * 100, 1)
    draw = np.round(outcomes.count([0, 0]) / len(outcomes) * 100, 1)
    invalid_1 = outcomes.count([None, 0])
    invalid_2 = outcomes.count([0, None])

    print(f"Agent 1 thắng: {win_1}%")
    print(f"Agent 2 thắng: {win_2}%")
    print(f"Hoà: {draw}%")
    print(f"Số ván Agent 1 đi nước không hợp lệ: {invalid_1}")
    print(f"Số ván Agent 2 đi nước không hợp lệ: {invalid_2}")


# So sánh agent_middle với agent_random qua 50 ván
get_win_percentages(agent_middle, agent_random, n_rounds=50)


> 🎯 **Kỳ vọng:** `agent_middle` nên thắng nhiều hơn `agent_random`, vì trong Connect Four, đánh vào cột
> giữa thường tạo ra nhiều cơ hội thắng hơn (cột giữa nằm trong nhiều đường thắng tiềm năng nhất — ngang, dọc,
> chéo). Đây là một ví dụ đơn giản cho thấy **heuristic tốt > heuristic ngẫu nhiên**, dù cả hai đều chưa phải
> RL thực sự.

### 2.6. Tự mình chơi thử với agent

Bạn có thể dùng `env.play()` để tự chơi (bằng cách click chuột) đối đầu với agent do mình viết. Truyền
`None` vào vị trí của người chơi thật.

> ⚠️ Chức năng này cần chạy trong môi trường Jupyter hỗ trợ widget tương tác (Kaggle Notebook, JupyterLab
> classic...). Nếu không hiển thị được, bạn vẫn có thể bỏ qua bước này — không ảnh hưởng tới các phần khác.

In [ ]:
# Bạn (None) chơi trước, đối đầu với agent_middle đi sau
env.play([None, agent_middle], width=500, height=450)


## Phần 3 — Bài tập thực hành

### Bài tập 1: Viết agent "chặn đối thủ 1 nước"

Viết một agent tên `agent_block` với chiến lược:

1. Nếu có nước đi giúp **agent thắng ngay lập tức** → đi nước đó.
2. Nếu không, nếu đối thủ sắp thắng ở nước tiếp theo → **chặn** bằng cách đánh vào cột đó.
3. Nếu không rơi vào 2 trường hợp trên → ưu tiên cột giữa, nếu không thì random.

> 💡 Gợi ý: bạn cần viết một hàm phụ mô phỏng "nếu đánh vào cột X thì có tạo ra `inarow` quân liên tiếp
> không?" — đây chính là ý tưởng cốt lõi của **one-step lookahead**, sẽ được học kỹ hơn ở bài tiếp theo
> của khoá học gốc.

In [ ]:
def check_win_if_drop(board, col, mark, config):
    """
    Trả về True nếu thả quân `mark` vào cột `col` sẽ tạo ra inarow quân liên tiếp
    (mô phỏng thử, không thay đổi board thật).
    """
    rows, columns, inarow = config.rows, config.columns, config.inarow
    board = board[:]  # copy để không sửa board gốc

    # Tìm hàng trống thấp nhất trong cột col
    row = None
    for r in range(rows - 1, -1, -1):
        if board[r * columns + col] == 0:
            row = r
            break
    if row is None:
        return False  # cột đã đầy

    board[row * columns + col] = mark

    def count_dir(dr, dc):
        r, c = row + dr, col + dc
        count = 0
        while 0 <= r < rows and 0 <= c < columns and board[r * columns + c] == mark:
            count += 1
            r += dr
            c += dc
        return count

    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dr, dc in directions:
        total = 1 + count_dir(dr, dc) + count_dir(-dr, -dc)
        if total >= inarow:
            return True
    return False


def agent_block(obs, config):
    valid_moves = [c for c in range(config.columns) if obs.board[c] == 0]
    my_mark = obs.mark
    opp_mark = 2 if my_mark == 1 else 1

    # 1) Có nước thắng ngay không?
    for col in valid_moves:
        if check_win_if_drop(obs.board, col, my_mark, config):
            return col

    # 2) Có cần chặn đối thủ không?
    for col in valid_moves:
        if check_win_if_drop(obs.board, col, opp_mark, config):
            return col

    # 3) Ưu tiên cột giữa, nếu không thì random
    mid = config.columns // 2
    if mid in valid_moves:
        return mid
    return random.choice(valid_moves)


print("Đã định nghĩa agent_block. Hãy chạy cell bên dưới để kiểm tra!")


In [ ]:
# TODO (bạn tự làm): So sánh agent_block với agent_random và agent_middle
# Gợi ý dùng hàm get_win_percentages đã viết ở trên

get_win_percentages(agent_block, agent_random, n_rounds=50)
print("-" * 40)
get_win_percentages(agent_block, agent_middle, n_rounds=50)


### Bài tập 2 (mở rộng, tự làm thêm)

1. Thử tăng `n_rounds` lên 200 và quan sát xem tỷ lệ thắng của `agent_block` có ổn định không.
2. Thử đổi `config` (ví dụ `rows=7, columns=8, inarow=5`) rồi chạy lại — `agent_block` có còn hoạt động
   đúng không? (Đây là bước kiểm tra tính **tổng quát hoá** của code — rất quan trọng khi sau này bạn viết
   môi trường tuỳ chỉnh cho giao lộ trong đồ án.)
3. Thử dùng `env.render(mode="ansi")` để in ra vài bước đầu của 1 ván giữa `agent_block` và `agent_random`,
   quan sát xem `agent_block` có thực sự chặn đúng lúc không.

## Phần 4 — Liên hệ với đồ án PPO/MAPPO điều khiển đèn giao thông

Bài học ConnectX tuy đơn giản nhưng dạy đúng **bộ khung tư duy** bạn sẽ dùng lại trong đồ án:

| Trong ConnectX | Trong đồ án điều khiển đèn giao thông |
|---|---|
| `env = make("connectx")` | Môi trường mô phỏng giao lộ (SUMO / Unity Digital Twin) |
| `obs.board` (trạng thái bàn cờ) | Trạng thái giao lộ: mật độ xe, độ dài hàng đợi, pha đèn hiện tại (từ YOLOv8 + ByteTrack) |
| `config` (columns, rows, inarow) | Tham số môi trường: số làn, số pha đèn, thời gian tối thiểu/tối đa mỗi pha |
| `action` (chọn cột) | Action: chọn pha đèn tiếp theo hoặc giữ nguyên pha hiện tại |
| Agent rule-based (`agent_middle`, `agent_block`) | Baseline rule-based (ví dụ: fixed-time hoặc actuated control) để so sánh với PPO/MAPPO |
| `evaluate()` — đấu nhiều ván, tính tỷ lệ thắng | Đánh giá agent PPO/MAPPO qua nhiều episode mô phỏng, đo các metric: thời gian chờ trung bình, throughput, độ dài hàng đợi |
| Agent RL học từ dữ liệu tự chơi (self-play) | PPO agent học từ trải nghiệm tương tác với môi trường mô phỏng, có thể "khởi động ấm" (warm-start) bằng Behavior Cloning |

> ✅ **Gợi ý thực hành:** trước khi code PPO/MAPPO đầy đủ, bạn có thể áp dụng đúng quy trình của bài học
> này cho môi trường giao thông của mình: (1) viết 1–2 agent rule-based đơn giản (baseline), (2) viết hàm
> đánh giá agent qua nhiều episode giống `get_win_percentages`, (3) sau đó mới thay agent rule-based bằng
> policy PPO đã huấn luyện và so sánh kết quả. Cách làm tuần tự này giúp bạn tách bạch được lỗi ở
> **môi trường** và lỗi ở **thuật toán RL** khi debug.

---

## Tóm tắt bài học

- ✅ Hiểu vòng lặp cơ bản của RL: `environment ↔ agent` qua `observation`, `action`, `reward`.
- ✅ Biết dùng `kaggle_environments` để tạo môi trường, chạy ván đấu, render kết quả.
- ✅ Viết được agent rule-based đơn giản (`obs, config -> action`).
- ✅ Biết đánh giá agent một cách khách quan qua nhiều ván đấu (`evaluate`).
- ✅ Liên hệ được các khái niệm này với bài toán RL thực tế trong đồ án tốt nghiệp.

**Bài tiếp theo trong khoá học gốc:** *One-Step Lookahead* — xây dựng agent dùng hàm heuristic để đánh giá
nước đi tốt hơn, là bước đệm tiến tới Minimax và sau đó là Deep Reinforcement Learning.